In [1]:
# download embedding model from hugging face
!pip install -U FlagEmbedding

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 10.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [FlagEmbedding]


In [5]:
import os
os.chdir("/content/drive/MyDrive/RAG-based-AI-Teaching-Assistant")
print("current dir :-", os.getcwd())

current dir :- /content/drive/MyDrive/RAG-based-AI-Teaching-Assistant


In [2]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3',
                       use_fp32=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

# text = "My name is Sujoy Das"
# embeddings = model.encode(text)['dense_vecs']
def create_embedding(text):
  embeddings = model.encode(text)['dense_vecs']
  return embeddings

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [15]:
import numpy as np
import json
import pandas as pd

In [11]:
# Test using two simple sentense
embeddings = create_embedding(["My name is Sujoy Das","Currently I am studing BCA in Midnapore College"])

# to see all numerical value
np.set_printoptions(threshold=np.inf)

print(embeddings)
print(embeddings.shape)

[[-3.2562e-02  1.3489e-02 -6.0547e-02 -5.2929e-04 -2.2781e-02 -6.4209e-02
   2.5146e-02 -5.4749e-02  1.4656e-02  3.1708e-02  1.7273e-02  4.3121e-02
  -2.3193e-02 -1.9241e-02  2.2690e-02  9.5444e-03  6.4453e-02 -5.8632e-03
  -3.7506e-02 -2.5620e-02  8.1558e-03  1.4748e-02  5.3619e-02  1.5154e-03
   2.0920e-02  3.4027e-02 -4.2999e-02 -1.8997e-02  3.5645e-02  4.8103e-03
   3.7079e-02  2.0889e-02  4.2053e-02 -3.2562e-02 -3.3051e-02 -8.4763e-03
   1.1803e-02 -1.6861e-02 -3.0167e-02  8.6823e-03  3.1738e-02  1.4145e-02
   3.7750e-02 -5.3650e-02 -1.4191e-02 -2.1927e-02 -2.0126e-02 -1.2741e-03
  -2.7084e-02  4.8256e-03  1.6113e-02 -2.6245e-02  5.8777e-02  1.5821e-03
   5.9204e-02  3.7750e-02  7.1869e-03 -3.0045e-02 -1.6317e-03  3.5915e-03
  -1.1787e-03  5.0187e-05  1.0345e-02  2.9240e-03  3.1342e-02  9.9426e-02
   5.7526e-02  6.0852e-02 -1.0529e-02  1.6495e-02 -2.4765e-02  9.5215e-03
  -2.6871e-02  9.7733e-03 -5.7007e-02 -1.5022e-02 -6.1035e-03 -1.7731e-02
  -2.8885e-02  1.1475e-02  8.7463e-02 

In [20]:
jsons = os.listdir("jsons")
my_dicts = []
chunks_id = 0
for json_file in jsons:
  with open(f"jsons/{json_file}","r")  as f:
    content = json.load(f)
  embeddings = create_embedding([c["text"] for c in content["chunks"]])
  print(f"Creating embedding for {json_file} ",end="")
  for i, chunk in enumerate(content["chunks"]):
    chunk["chunks_id"] = chunks_id
    chunks_id+=1
    chunk["embedding"] = embeddings[i]
    my_dicts.append(chunk)
  # break
  print("==> done")
# print(my_dicts)

Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00,  4.54it/s]


Creating embedding for 2_heading_paragraphs_and_links.mp3.json ==> done


Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00,  6.42it/s]


Creating embedding for 3_image_lists_and_tables.mp3.json ==> done


Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00,  4.43it/s]


Creating embedding for 4_seo_and_core_web_vitals.mp3.json ==> done
Creating embedding for 5_forms_and_input_tags.mp3.json ==> done


Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00,  4.10it/s]

Creating embedding for 1_basic_structure_of_an_html_website.mp3.json ==> done


In [21]:
df = pd.DataFrame.from_records(my_dicts)
print(df.head(3))

  number                         title  start   end  \
0      2  heading_paragraphs_and_links   0.00  3.20   
1      2  heading_paragraphs_and_links   3.20  6.36   
2      2  heading_paragraphs_and_links   6.36  8.32   

                                                text  chunks_id  \
0   In today's video, I will tell you about headi...          0   
1   Along with that, I will also tell you about a...          1   
2    If you have found this course awesome till now,          2   

                                           embedding  
0  [-0.03574, 0.01277, -0.01043, 0.03735, -0.0352...  
1  [-0.07, 0.01678, -0.0245, 0.02563, -0.0358, -0...  
2  [-0.03506, 0.02206, -0.02187, -0.01135, -0.025...  


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2316 entries, 0 to 2315
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   number     2316 non-null   object 
 1   title      2316 non-null   object 
 2   start      2316 non-null   float64
 3   end        2316 non-null   float64
 4   text       2316 non-null   object 
 5   chunks_id  2316 non-null   int64  
 6   embedding  2316 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 126.8+ KB
